In [ ]:
#Python
# Asegúrense de que esta línea esté en una celda anterior para instalar PySpark
#!pip install pyspark

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count

# 1. Crear una sesión Spark
spark = SparkSession.builder \
    .appName("SparkSQLExampleColab") \
    .master("local[*]") \
    .getOrCreate()

print("--- Ejemplo de Spark SQL y DataFrame ---")

# 2. Crear un DataFrame a partir de datos en memoria (simulando un archivo CSV)
data = [("Alice", 1, 30),
        ("Bob", 2, 25),
        ("Charlie", 1, 35),
        ("David", 2, 28),
        ("Eve", 1, 30)]

columns = ["name", "department_id", "age"]
df = spark.createDataFrame(data, columns)

print("DataFrame original:")
df.show()

# 3. Registrar el DataFrame como una vista temporal para poder consultarlo con SQL
df.createOrReplaceTempView("employees")

# 4. Ejecutar una consulta SQL
print("\nResultado de la consulta SQL (empleados mayores de 28):")
result_sql = spark.sql("SELECT name, age FROM employees WHERE age > 28")
result_sql.show()

# 5. Realizar las mismas operaciones usando la API de DataFrame
print("\nResultado de la API de DataFrame (empleados en el departamento 1):")
result_df_api = df.filter(col("department_id") == 1).select("name", "age")
result_df_api.show()

# 6. Agregación usando la API de DataFrame
print("\nConteo de empleados por departamento:")
df.groupBy("department_id").agg(count("*").alias("employee_count")).show()

# 7. Detener la sesión Spark
spark.stop()
print("\nSesión Spark detenida.")


Error: A JNI error has occurred, please check your installation and try again
Exception in thread "main" java.lang.UnsupportedClassVersionError: org/apache/spark/launcher/Main has been compiled by a more recent version of the Java Runtime (class file version 61.0), this version of the Java Runtime only recognizes class file versions up to 52.0
	at java.lang.ClassLoader.defineClass1(Native Method)
	at java.lang.ClassLoader.defineClass(ClassLoader.java:756)
	at java.security.SecureClassLoader.defineClass(SecureClassLoader.java:142)
	at java.net.URLClassLoader.defineClass(URLClassLoader.java:473)
	at java.net.URLClassLoader.access$100(URLClassLoader.java:74)
	at java.net.URLClassLoader$1.run(URLClassLoader.java:369)
	at java.net.URLClassLoader$1.run(URLClassLoader.java:363)
	at java.security.AccessController.doPrivileged(Native Method)
	at java.net.URLClassLoader.findClass(URLClassLoader.java:362)
	at java.lang.ClassLoader.loadClass(ClassLoader.java:418)
	at sun.misc.Launcher$AppClassLoad

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

# Spark Streaming + netcat (Colab)
Notebook con la estructura solicitada:

- **paso 1.** Instalar PySpark  
- **paso 2.** Ejecutar servidor socket tipo *netcat real* (muestra `Listening on port: ...`)  
- **paso 3.** Iniciar Spark Streaming y conectarse al socket  
- **paso 4.** (Opcional) Simular netcat dentro de Colab con una celda Bash (bloqueante)  

> Usa el **mismo puerto** en todos los pasos (por defecto `9999`).

In [2]:
# === paso 1: Instalar PySpark y verificar entorno ===
#!pip -q install pyspark==3.5.1 > /dev/null

import sys, platform, importlib, subprocess

def sh(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True)
    except subprocess.CalledProcessError as e:
        return e.output

print("Python:", sys.version.split()[0])
print("SO:", platform.platform())
print("Java:\n", sh("java -version"))

import pyspark
print("PySpark versión:", pyspark.__version__)
assert importlib.import_module("pyspark"), "No se pudo importar pyspark"
print("✅ pyspark importado correctamente")

Python: 3.12.3
SO: Linux-6.14.0-36-generic-x86_64-with-glibc2.39
Java:
 openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-124.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-124.04, mixed mode, sharing)

PySpark versión: 4.0.1
✅ pyspark importado correctamente


## paso 2. Ejecuten el código en una celda. **Mantener la experiencia “real” de netcat**

**Importante:** Después de ejecutar esta celda, verán un mensaje como:  
`Listening on port: 9999`  
Deberán **ejecutar un comando netcat en su propia máquina** o en otra instancia de terminal a la que Colab pueda acceder para enviar datos al socket.


In [3]:
# === paso 2: Servidor socket tipo "netcat real" ===
# Este servidor corre dentro de Colab y espera una conexión de netcat como CLIENTE.
# Ejemplo de conexión (en otra terminal o celda con !nc si está disponible):
#   nc 127.0.0.1 9999

import socket, threading

PORT = 9999

def start_netcat_like_server(port=PORT):
    srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    srv.bind(("127.0.0.1", port))
    srv.listen(1)
    print(f"Listening on port: {port}")  # 👈 Mensaje tal como en la nota
    conn, addr = srv.accept()
    print(f"Cliente conectado: {addr}")
    try:
        while True:
            data = conn.recv(4096)
            if not data:
                break
            print("Mensaje recibido (desde nc):", data.decode("utf-8").rstrip())
    finally:
        conn.close()
        srv.close()
        print("Servidor cerrado.")

# Iniciar el servidor Netcat en un hilo separado
netcat_thread = threading.Thread(target=start_netcat_like_server, args=(PORT,))
netcat_thread.daemon = True
netcat_thread.start()

Listening on port: 9999


## paso 3. Iniciar el StreamingContext y conectar al socket

Usa el **mismo puerto** del paso 2 (`9999`). Escribe en la terminal/celda donde tengas `nc` ejecutándose y verás los resultados en la salida de esta celda.


In [4]:
# === paso 3: Spark Streaming conectado al socket (30s) ===
from pyspark.sql import SparkSession
from pyspark.streaming import StreamingContext

spark = (SparkSession.builder
         .appName("DemoNetcatReal")
         .master("local[2]")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")

ssc = StreamingContext(sc, 1)
lines = ssc.socketTextStream("127.0.0.1", 9999)

words = lines.flatMap(lambda line: line.strip().split())
pairs = words.map(lambda w: (w.lower(), 1))
counts = pairs.reduceByKey(lambda a, b: a + b)

counts.pprint()

print("\nIniciando Spark Streaming context. Conectándose a localhost:9999...")
print("Escribe mensajes en la terminal/celda donde ejecutaste 'nc' y presiona Enter.")
ssc.start()

try:
    ssc.awaitTerminationOrTimeout(30)
except KeyboardInterrupt:
    print("Spark Streaming context detenido por el usuario.")
finally:
    ssc.stop(stopSparkContext=False, stopGracefully=True)
    print("\nSesión Spark y StreamingContext detenidos.")

Error: A JNI error has occurred, please check your installation and try again
Exception in thread "main" java.lang.UnsupportedClassVersionError: org/apache/spark/launcher/Main has been compiled by a more recent version of the Java Runtime (class file version 61.0), this version of the Java Runtime only recognizes class file versions up to 52.0
	at java.lang.ClassLoader.defineClass1(Native Method)
	at java.lang.ClassLoader.defineClass(ClassLoader.java:756)
	at java.security.SecureClassLoader.defineClass(SecureClassLoader.java:142)
	at java.net.URLClassLoader.defineClass(URLClassLoader.java:473)
	at java.net.URLClassLoader.access$100(URLClassLoader.java:74)
	at java.net.URLClassLoader$1.run(URLClassLoader.java:369)
	at java.net.URLClassLoader$1.run(URLClassLoader.java:363)
	at java.security.AccessController.doPrivileged(Native Method)
	at java.net.URLClassLoader.findClass(URLClassLoader.java:362)
	at java.lang.ClassLoader.loadClass(ClassLoader.java:418)
	at sun.misc.Launcher$AppClassLoad

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

## paso 4 (Más fácil en Colab). Simular el servidor de Netcat dentro de Colab (bloqueante)

> **ATENCIÓN:** Esta celda **bloqueará** mientras el servidor esté activo.  
> Úsala solo si no puedes abrir `nc` aparte. **Ejecuta esta celda después de `ssc.start()`** y asegúrate de que el puerto coincida (`9999`).  
> Escribe texto y presiona Enter para enviarlo al socket.


In [ ]:
%%bash
# === paso 4: Simular "netcat -l -p 9999" dentro de Colab (bloqueante) ===
while true; do read line; echo "$line"; done | nc -l -p 9999

In [ ]:
# Asegúrense de que esta línea esté en una celda anterior para instalar PySpark
# !pip install pyspark

from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator
# 1. Crear una sesión Spark
spark = SparkSession.builder \
    .appName("SparkMLlibExampleColab") \
    .master("local[*]") \
    .getOrCreate()

print("--- Ejemplo de Spark MLlib con la API de DataFrame (spark.ml) ---")
# 2. Simular datos de entrenamiento (ejemplo simple de clasificación binaria)
# Features: "feature1", "feature2"
# Label: "label" (0 o 1)
data = [(1.0, 2.0, 0),
        (2.0, 3.0, 0),
        (3.0, 4.0, 1),
        (4.0, 5.0, 1),
        (5.0, 6.0, 1),
        (6.0, 7.0, 0)] # Un dato más para ilustrar

columns = ["feature1", "feature2", "label"]
df = spark.createDataFrame(data, columns)
print("DataFrame de entrada:")
df.show()

# 3. VectorAssembler: Combina las columnas de características en un solo vector.
assembler = VectorAssembler(inputCols=["feature1", "feature2"], outputCol="features")

# 4. StandardScaler: Escala las características para normalizarlas.
# Nota: Este paso es opcional pero recomendado para muchos algoritmos de ML.
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)

# 5. LogisticRegression: El algoritmo de clasificación.
lr = LogisticRegression(featuresCol="scaledFeatures", labelCol="label", maxIter=10)

# 6. Crear un Pipeline
# El orden de los stages es importante
pipeline = Pipeline(stages=[assembler, scaler, lr])

# 7. Entrenar el modelo
print("\nEntrenando el modelo...")
model = pipeline.fit(df)
print("Modelo entrenado.")

# 8. Realizar predicciones en el mismo DataFrame (para demostración)
# En un escenario real, usarías un nuevo DataFrame de datos de prueba
predictions = model.transform(df)

print("\nPredicciones:")
predictions.select("feature1", "feature2", "label", "prediction", "probability").show()

# 9. Evaluar el modelo (para clasificación binaria, podemos usar la curva ROC)
evaluator = BinaryClassificationEvaluator(labelCol="label")
auc = evaluator.evaluate(predictions)
print(f"\nÁrea bajo la curva ROC: {auc}")

# Detener la sesión Spark
spark.stop()
print("\nSesión Spark detenida.")
